# AI-Powered Infrared Temperature Prediction System

## Business Problem

Industrial equipment such as motors, transformers, and electrical panels can overheat during operation. Overheating often leads to equipment failure, production downtime, increased maintenance costs, and safety risks.

Infrared thermography provides a non-contact method of measuring surface temperatures. By leveraging machine learning, this project aims to predict object temperatures from infrared sensor measurements, enabling predictive maintenance and early fault detection.

## Project Objectives

- Understand the infrared thermography dataset.
- Perform exploratory data analysis.
- Clean and preprocess the data.
- Build multiple machine learning regression models.
- Evaluate and compare model performance.
- Deploy the best model using Streamlit.

## Technologies

- Python
- Pandas
- NumPy
- Matplotlib
- Plotly
- Scikit-learn
- XGBoost
- LightGBM
- SHAP
- Streamlit

In [3]:
# ============================================
# Import Required Libraries
# ============================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from ucimlrepo import fetch_ucirepo

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [5]:
# ============================================
# Fetch Dataset
# ============================================

dataset = fetch_ucirepo(id=925)

X = dataset.data.features
y = dataset.data.targets

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [6]:
# Inspect Dataset
print(type(X))
print(type(y))

<class 'pandas.DataFrame'>
<class 'pandas.DataFrame'>


In [7]:
X.head()

,Gender,Age,Ethnicity,T_atm,Humidity,Distance,T_offset1,Max1R13_1,Max1L13_1,aveAllR13_1,aveAllL13_1,T_RC1,T_RC_Dry1,T_RC_Wet1,T_RC_Max1,T_LC1,T_LC_Dry1,T_LC_Wet1,T_LC_Max1,RCC1,LCC1,canthiMax1,canthi4Max1,T_FHCC1,T_FHRC1,T_FHLC1,T_FHBC1,T_FHTC1,T_FH_Max1,T_FHC_Max1,T_Max1,T_OR1,T_OR_Max1
0,Male,41-50,White,24.0,28.0,0.8,0.7025,35.0300,35.3775,34.4000,34.9175,34.9850,34.9850,34.7625,35.0325,35.3375,35.3375,34.4850,35.3775,34.7850,34.4650,35.3775,35.3375,33.5775,33.4775,33.3725,33.4925,33.0025,34.5300,34.0075,35.6925,35.6350,35.6525
1,Female,31-40,Black or African-American,24.0,26.0,0.8,0.7800,34.5500,34.5200,33.9300,34.2250,34.7100,34.6325,34.6400,34.7425,34.5600,34.5375,34.3500,34.5750,34.3225,34.2400,34.7400,34.7150,34.0325,34.0550,33.6775,33.9700,34.0025,34.6825,34.6600,35.1750,35.0925,35.1075
2,Female,21-30,White,24.0,26.0,0.8,0.8625,35.6525,35.5175,34.2775,34.8000,35.6850,35.6675,35.6150,35.7175,35.5025,35.5025,35.2950,35.5300,35.3575,35.0925,35.7175,35.6825,34.9000,34.8275,34.6475,34.8200,34.6700,35.3450,35.2225,35.9125,35.8600,35.8850
3,Female,21-30,Black or African-American,24.0,27.0,0.8,0.9300,35.2225,35.6125,34.3850,35.2475,35.2075,35.2000,35.1175,35.2250,35.5950,35.5950,35.3275,35.6125,34.9100,35.1700,35.6125,35.5950,34.4400,34.4225,34.6550,34.3025,34.9175,35.6025,35.3150,35.7200,34.9650,34.9825
4,Male,18-20,White,24.0,27.0,0.8,0.8950,35.5450,35.6650,34.9100,35.3675,35.6025,35.4750,35.5700,35.6400,35.6400,35.6400,35.0775,35.6675,35.3550,35.1200,35.6650,35.6475,35.0900,35.1600,34.3975,34.6700,33.8275,35.4175,35.3725,35.8950,35.5875,35.6175


In [8]:
y.head()

,aveOralF,aveOralM
0,36.85,36.59
1,37.00,37.19
2,37.20,37.34
3,36.85,37.09
4,36.80,37.04


## Merge Features and Target

To simplify data exploration and preprocessing, the feature matrix and target variable are combined into a single DataFrame.

In [19]:
# Merge Dataset
df_raw = pd.concat([X, y], axis=1)

df_raw.head()

,Gender,Age,Ethnicity,T_atm,Humidity,Distance,T_offset1,Max1R13_1,Max1L13_1,aveAllR13_1,aveAllL13_1,T_RC1,T_RC_Dry1,T_RC_Wet1,T_RC_Max1,T_LC1,T_LC_Dry1,T_LC_Wet1,T_LC_Max1,RCC1,LCC1,canthiMax1,canthi4Max1,T_FHCC1,T_FHRC1,T_FHLC1,T_FHBC1,T_FHTC1,T_FH_Max1,T_FHC_Max1,T_Max1,T_OR1,T_OR_Max1,aveOralF,aveOralM
0,Male,41-50,White,24.0,28.0,0.8,0.7025,35.0300,35.3775,34.4000,34.9175,34.9850,34.9850,34.7625,35.0325,35.3375,35.3375,34.4850,35.3775,34.7850,34.4650,35.3775,35.3375,33.5775,33.4775,33.3725,33.4925,33.0025,34.5300,34.0075,35.6925,35.6350,35.6525,36.85,36.59
1,Female,31-40,Black or African-American,24.0,26.0,0.8,0.7800,34.5500,34.5200,33.9300,34.2250,34.7100,34.6325,34.6400,34.7425,34.5600,34.5375,34.3500,34.5750,34.3225,34.2400,34.7400,34.7150,34.0325,34.0550,33.6775,33.9700,34.0025,34.6825,34.6600,35.1750,35.0925,35.1075,37.00,37.19
2,Female,21-30,White,24.0,26.0,0.8,0.8625,35.6525,35.5175,34.2775,34.8000,35.6850,35.6675,35.6150,35.7175,35.5025,35.5025,35.2950,35.5300,35.3575,35.0925,35.7175,35.6825,34.9000,34.8275,34.6475,34.8200,34.6700,35.3450,35.2225,35.9125,35.8600,35.8850,37.20,37.34
3,Female,21-30,Black or African-American,24.0,27.0,0.8,0.9300,35.2225,35.6125,34.3850,35.2475,35.2075,35.2000,35.1175,35.2250,35.5950,35.5950,35.3275,35.6125,34.9100,35.1700,35.6125,35.5950,34.4400,34.4225,34.6550,34.3025,34.9175,35.6025,35.3150,35.7200,34.9650,34.9825,36.85,37.09
4,Male,18-20,White,24.0,27.0,0.8,0.8950,35.5450,35.6650,34.9100,35.3675,35.6025,35.4750,35.5700,35.6400,35.6400,35.6400,35.0775,35.6675,35.3550,35.1200,35.6650,35.6475,35.0900,35.1600,34.3975,34.6700,33.8275,35.4175,35.3725,35.8950,35.5875,35.6175,36.80,37.04


In [20]:
df = df_raw.copy()

In [ ]:
# Dataset shape
print(f"Rows: {df_raw.shape[0]}")
print(f"Columns: {df_raw.shape[1]}")

Rows: 1020
Columns: 35


In [12]:
# Column Name
df.columns

Index(['Gender', 'Age', 'Ethnicity', 'T_atm', 'Humidity', 'Distance',
       'T_offset1', 'Max1R13_1', 'Max1L13_1', 'aveAllR13_1', 'aveAllL13_1',
       'T_RC1', 'T_RC_Dry1', 'T_RC_Wet1', 'T_RC_Max1', 'T_LC1', 'T_LC_Dry1',
       'T_LC_Wet1', 'T_LC_Max1', 'RCC1', 'LCC1', 'canthiMax1', 'canthi4Max1',
       'T_FHCC1', 'T_FHRC1', 'T_FHLC1', 'T_FHBC1', 'T_FHTC1', 'T_FH_Max1',
       'T_FHC_Max1', 'T_Max1', 'T_OR1', 'T_OR_Max1', 'aveOralF', 'aveOralM'],
      dtype='str')

In [13]:
# Data type
df.dtypes

Gender             str
Age                str
Ethnicity          str
T_atm          float64
Humidity       float64
Distance       float64
T_offset1      float64
Max1R13_1      float64
Max1L13_1      float64
aveAllR13_1    float64
aveAllL13_1    float64
T_RC1          float64
T_RC_Dry1      float64
T_RC_Wet1      float64
T_RC_Max1      float64
T_LC1          float64
T_LC_Dry1      float64
T_LC_Wet1      float64
T_LC_Max1      float64
RCC1           float64
LCC1           float64
canthiMax1     float64
canthi4Max1    float64
T_FHCC1        float64
T_FHRC1        float64
T_FHLC1        float64
T_FHBC1        float64
T_FHTC1        float64
T_FH_Max1      float64
T_FHC_Max1     float64
T_Max1         float64
T_OR1          float64
T_OR_Max1      float64
aveOralF       float64
aveOralM       float64
dtype: object

In [14]:
# General information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 35 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Gender       1020 non-null   str    
 1   Age          1020 non-null   str    
 2   Ethnicity    1020 non-null   str    
 3   T_atm        1020 non-null   float64
 4   Humidity     1020 non-null   float64
 5   Distance     1018 non-null   float64
 6   T_offset1    1020 non-null   float64
 7   Max1R13_1    1020 non-null   float64
 8   Max1L13_1    1020 non-null   float64
 9   aveAllR13_1  1020 non-null   float64
 10  aveAllL13_1  1020 non-null   float64
 11  T_RC1        1020 non-null   float64
 12  T_RC_Dry1    1020 non-null   float64
 13  T_RC_Wet1    1020 non-null   float64
 14  T_RC_Max1    1020 non-null   float64
 15  T_LC1        1020 non-null   float64
 16  T_LC_Dry1    1020 non-null   float64
 17  T_LC_Wet1    1020 non-null   float64
 18  T_LC_Max1    1020 non-null   float64
 19  RCC1         1020

In [15]:
# Statistical Summary
df.describe().T

,count,mean,std,min,25%,50%,75%,max
T_atm,1020.0,24.115392,1.336338,20.2000,23.400000,24.000000,24.700000,29.1000
Humidity,1020.0,28.723039,13.071627,9.9000,17.600000,26.300000,36.200000,61.2000
Distance,1018.0,0.729784,2.456486,0.5400,0.600000,0.620000,0.700000,79.0000
T_offset1,1020.0,0.968648,0.362587,-0.5900,0.772500,0.940000,1.140000,2.8750
Max1R13_1,1020.0,35.596533,0.574888,33.8975,35.247500,35.548750,35.872500,38.4050
Max1L13_1,1020.0,35.611474,0.549760,34.1225,35.271875,35.575000,35.883125,38.0425
aveAllR13_1,1020.0,34.888475,0.718613,31.7700,34.456250,34.915000,35.300000,37.5750
aveAllL13_1,1020.0,35.011345,0.633836,32.9025,34.651250,34.997500,35.363125,37.6800
T_RC1,1020.0,35.659921,0.553897,33.9850,35.332500,35.602500,35.910625,38.3850
T_RC_Dry1,1020.0,35.587143,0.569278,33.8250,35.249375,35.533750,35.855625,38.3800


In [21]:
# It summarise both numeric and categorical data
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Gender,1020,2,Female,606,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,1020,8,18-20,534,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ethnicity,1020,6,White,506,NaN,NaN,NaN,NaN,NaN,NaN,NaN
T_atm,1020.0,NaN,NaN,NaN,24.115392,1.336338,20.2,23.4,24.0,24.7,29.1
Humidity,1020.0,NaN,NaN,NaN,28.723039,13.071627,9.9,17.6,26.3,36.2,61.2
Distance,1018.0,NaN,NaN,NaN,0.729784,2.456486,0.54,0.6,0.62,0.7,79.0
T_offset1,1020.0,NaN,NaN,NaN,0.968648,0.362587,-0.59,0.7725,0.94,1.14,2.875
Max1R13_1,1020.0,NaN,NaN,NaN,35.596533,0.574888,33.8975,35.2475,35.54875,35.8725,38.405
Max1L13_1,1020.0,NaN,NaN,NaN,35.611474,0.54976,34.1225,35.271875,35.575,35.883125,38.0425
aveAllR13_1,1020.0,NaN,NaN,NaN,34.888475,0.718613,31.77,34.45625,34.915,35.3,37.575


In [16]:
# Metadata
# It Explains:
# Dataset source
# Purpose
# Target variable(s)
# Collection method

dataset.metadata

{'uci_id': 925,
 'name': 'Infrared Thermography Temperature',
 'repository_url': 'https://archive.ics.uci.edu/dataset/925/infrared+thermography+temperature+dataset',
 'data_url': 'https://archive.ics.uci.edu/static/public/925/data.csv',
 'abstract': 'The Infrared Thermography Temperature Dataset contains temperatures read from various locations of inferred images about patients, with the addition of oral temperatures measured for each individual. The 33 features consist of gender, age, ethnicity, ambiant temperature, humidity, distance, and other temperature readings from the thermal images. The dataset is intended to be used in a regression task to predict the oral temperature using the environment information as well as the thermal image readings. ',
 'area': 'Health and Medicine',
 'tasks': ['Regression'],
 'characteristics': ['Tabular'],
 'num_instances': 1020,
 'num_features': 33,
 'feature_types': ['Real', 'Categorical'],
 'demographics': ['Gender', 'Age', 'Ethnicity'],
 'target_

In [1]:
# Variable Information
dataset.variables

NameError: name 'dataset' is not defined

In [18]:
df.isnull().sum()

Gender         0
Age            0
Ethnicity      0
T_atm          0
Humidity       0
Distance       2
T_offset1      0
Max1R13_1      0
Max1L13_1      0
aveAllR13_1    0
aveAllL13_1    0
T_RC1          0
T_RC_Dry1      0
T_RC_Wet1      0
T_RC_Max1      0
T_LC1          0
T_LC_Dry1      0
T_LC_Wet1      0
T_LC_Max1      0
RCC1           0
LCC1           0
canthiMax1     0
canthi4Max1    0
T_FHCC1        0
T_FHRC1        0
T_FHLC1        0
T_FHBC1        0
T_FHTC1        0
T_FH_Max1      0
T_FHC_Max1     0
T_Max1         0
T_OR1          0
T_OR_Max1      0
aveOralF       0
aveOralM       0
dtype: int64

In [22]:
# Save the original dataset
df.to_csv("../data/raw.csv", index=False)

print("Raw dataset saved successfully!")

Raw dataset saved successfully!


## Initial Findings

This notebook introduced the Infrared Thermography Temperature dataset and provided an overview of its structure.

Key observations include:

- Number of observations
- Number of features
- Target variable(s)
- Data types
- Initial statistical summary

These findings will guide the data cleaning and exploratory data analysis in the next notebook.